# Creating a Mean Reversion strategy

In this video, we are going to create a professional mean reversion strategy

Want to show how mean reversion is traded professionally as what you see on the internet is not 100% accurate.

This is a strategy that you can trade manually or automatically.

I'm a strong advocate of simplicity and this strategy is not going to use any machine-learning nor advanced mathematics.

It's just going to be using basic econometrics.

Financial data is extremely noisy yet there are statistical patterns that exhibit inside that are not visible to the naked eye.
There's a misconception that you cannot make money with basic strategies because the markets are complex - this is futher from the truth.

## Goal = Find a statistical pattern

![Alt text](https://i.postimg.cc/023pdYK0/BCH.png)

In [ ]:
import numpy as np
import pandas as pd

We have Bitcoin Cash data. I know nothing about this project or coin or the history. All that I am interested in is its price movements.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Sample OHLC data ships with the repo (data/samples/). Resolve it whether the
# notebook runs from the repo root (VS Code default) or from its own folder.
csv_name = 'mean_reversion_ohlc.csv'
csv_path = next(p for p in (Path('data/samples') / csv_name,
                            Path('../../data/samples') / csv_name) if p.exists())
df = pd.read_csv(csv_path)
del df['Unnamed: 0']
df


## Create Log Returns

In [ ]:
df['close_log_return'] = np.log(df['c']/df['c'].shift(1))
df

## Create Auto-Regressive Log Returns

In [ ]:
df['close_log_return_lag_1'] = df['close_log_return'].shift(1)
df = df.dropna()
df

## Encode Direction

* 1 => Long => Bet up
* -1 => Short => Bet down

In [ ]:
for col in ['close_log_return','close_log_return_lag_1']:
    df[f'{col}_dir'] = df[col].map(lambda x: 1 if x > 0 else -1)
df

## Study Auto-Regressive Price Movements

In [ ]:
df.groupby('close_log_return_lag_1_dir').aggregate({'close_log_return':['mean','count', 'sum']})

## How to interpret this?

Well it exhibits strong mean reversion behaviour

1. If the previous price movement went down, it has a strong price movement that it will go up in the next day.
2. If the previous price movement went up, it has a strong price movement that it will go down in the next day.

## Out-of-Sample Validation

To determine whether the observed pattern persists over time rather than being a temporary effect, the statistic is evaluated on a holdout dataset consisting of the most recent observations.

This ensures that the results generalize beyond the in-sample period.

In [ ]:
i = int(len(df) * 0.75)

in_sample, out_sample = df.iloc[:i], df.iloc[i:]

in_sample.groupby('close_log_return_lag_1_dir').aggregate({'close_log_return':['mean','count', 'sum']})

In [ ]:
out_sample.groupby('close_log_return_lag_1_dir').aggregate({'close_log_return':['mean','count', 'sum']})

We can see the pattern persists in the out-of-sample so this means that the mean reversion behaviour persists with the most recent data.

## Display Equity Curve

We are interested to know what the equity curve looks like if we traded this mean reversion strategy.

In [ ]:
df['signal'] = -1 * df['close_log_return_lag_1_dir']
df

In [ ]:
df['trade_log_return'] = df['signal'] * df['close_log_return']
df

In [ ]:
df['trade_log_return'].cumsum().plot()

## Strategy Statistics

Win Rate

In [ ]:
df['is_won'] = df['trade_log_return'] > 0
df['is_won'].mean()

Total Gross Compound Return

In [ ]:
r = np.exp(df['trade_log_return'].sum()) - 1
r

In [ ]:
12 * r

Annualized Sharpe

This dataset uses **daily (1d)** bars, so there are `365` periods per year (crypto trades every day). The per-period Sharpe is scaled by `sqrt(365)` to annualize.


In [ ]:
df['trade_log_return'].mean() / df['trade_log_return'].std() * np.sqrt(365)

## Add Round-Trip Fees

In [ ]:
df['cum_trade_log_return'] = df['trade_log_return'].cumsum()
df

In [ ]:
capital = 1000
df['post_trade_notional_value'] = capital + df['cum_trade_log_return'] * capital
df

In [ ]:
df['pre_trade_notional_value'] = df['post_trade_notional_value'].shift()
df

In [ ]:
df['pre_trade_notional_value'] = df['pre_trade_notional_value'].fillna(capital)
df

In [ ]:
TAKER_FEE_BPS = 2.0
MAKER_FEE_BPS = 1.5

def fee_bps(bp):
  return bp / 10000

TAKER_FEE = fee_bps(TAKER_FEE_BPS)
MAKER_FEE = fee_bps(MAKER_FEE_BPS)
df['entry_fee'] = df['pre_trade_notional_value'] * MAKER_FEE
df

In [ ]:
df['exit_fee'] = df['post_trade_notional_value'] * MAKER_FEE
df

In [ ]:
df['roundtrip_fees'] = df['entry_fee'] + df['exit_fee']
df

In [ ]:
df['cum_roundtrip_fees'] = df['roundtrip_fees'].cumsum()
df

In [ ]:
df['net_equity'] = df['post_trade_notional_value'] - df['cum_roundtrip_fees']
df

In [ ]:
df['net_equity'].plot()

## Exercises

1. Add transaction fees
2. Display equity curve factoring in fees
3. Add trade sizing starting with $12